In [31]:
pip install langchain openai langsmith python-dotenv

In [15]:
!pip install langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.0 MB/s eta 0:00:00


In [17]:
from langchain_core.prompts import PromptTemplate

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import os

In [19]:
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract ONLY explicitly mentioned:
- Skills
- Experience
- Tools

Do NOT assume anything.

Resume:
{resume}

Return JSON format only.
"""
)

def extract(resume):
    return llm.invoke(extract_prompt.format(resume=resume)).content

In [20]:
match_prompt = PromptTemplate(
    input_variables=["resume_data", "job_desc"],
    template="""
Compare resume with job description.

Resume:
{resume_data}

Job:
{job_desc}

Return:
- matched_skills
- missing_skills
- match_percentage
"""
)

def match(resume_data, job_desc):
    return llm.invoke(match_prompt.format(
        resume_data=resume_data,
        job_desc=job_desc
    )).content

In [21]:
score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
Give a score (0-100).

Rules:
- High match → 80-100
- Medium → 50-79
- Low → <50

Match Data:
{match_data}

Return only number.
"""
)

def score(match_data):
    return llm.invoke(score_prompt.format(match_data=match_data)).content

In [22]:
explain_prompt = PromptTemplate(
    input_variables=["score", "match_data"],
    template="""
Explain why this score was given.

Score: {score}
Match: {match_data}

Do NOT assume missing skills.
"""
)

def explain(score_val, match_data):
    return llm.invoke(explain_prompt.format(
        score=score_val,
        match_data=match_data
    )).content

In [23]:
def run_pipeline(resume, job_desc):
    extracted = extract(resume)
    matched = match(extracted, job_desc)
    score_val = score(matched)
    explanation = explain(score_val, matched)

    return {
        "Extracted": extracted,
        "Match": matched,
        "Score": score_val,
        "Explanation": explanation
    }

In [24]:
job_desc = """
Looking for Data Scientist.
Skills: Python, Machine Learning, SQL, Pandas, NLP
Experience: 2+ years
"""

In [25]:
strong_resume = """
Data Scientist with 3 years experience.
Skills: Python, Machine Learning, SQL, Pandas, NLP
Tools: TensorFlow, Scikit-learn
"""

In [26]:
avg_resume = """
Data Analyst with 1 year experience.
Skills: Python, SQL, Excel
Tools: Pandas
"""

In [27]:
weak_resume = """
Fresher.
Skills: MS Word, Communication
"""

In [29]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [34]:
def extract(resume):
    res = llm.invoke(extract_prompt.format(resume=resume))
    return res.content if hasattr(res, "content") else str(res)

In [35]:
def match(resume_data, job_desc):
    res = llm.invoke(match_prompt.format(
        resume_data=resume_data,
        job_desc=job_desc
    ))
    return res.content if hasattr(res, "content") else str(res)

In [36]:
def score(match_data):
    res = llm.invoke(score_prompt.format(match_data=match_data))
    return res.content if hasattr(res, "content") else str(res)

In [37]:
def explain(score_val, match_data):
    res = llm.invoke(explain_prompt.format(
        score=score_val,
        match_data=match_data
    ))
    return res.content if hasattr(res, "content") else str(res)

In [38]:
def run_pipeline(resume, job_desc):
    extracted = extract(resume)
    print("Extracted:", extracted)

    matched = match(extracted, job_desc)
    print("Matched:", matched)

    score_val = score(matched)
    print("Score:", score_val)

    explanation = explain(score_val, matched)
    print("Explanation:", explanation)

    return {
        "Score": score_val,
        "Explanation": explanation
    }

In [52]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [54]:
def extract(resume):
    return {
        "skills": [word for word in ["Python", "Machine Learning", "SQL", "Pandas", "NLP"] if word in resume],
        "experience": "present" if "year" in resume else "none",
        "tools": [word for word in ["TensorFlow", "Scikit-learn"] if word in resume]
    }


def match(resume_data, job_desc):
    job_skills = ["Python", "Machine Learning", "SQL", "Pandas", "NLP"]

    matched = [skill for skill in job_skills if skill in resume_data["skills"]]
    missing = [skill for skill in job_skills if skill not in resume_data["skills"]]

    match_percentage = int((len(matched) / len(job_skills)) * 100)

    return {
        "matched_skills": matched,
        "missing_skills": missing,
        "match_percentage": match_percentage
    }


def score(match_data):
    percent = match_data["match_percentage"]

    if percent > 80:
        return 90
    elif percent > 50:
        return 70
    else:
        return 40


def explain(score_val, match_data):
    return f"""
Candidate scored {score_val} because:
Matched skills: {match_data['matched_skills']}
Missing skills: {match_data['missing_skills']}
Match Percentage: {match_data['match_percentage']}%
"""


def run_pipeline(resume, job_desc):
    extracted = extract(resume)
    matched = match(extracted, job_desc)
    score_val = score(matched)
    explanation = explain(score_val, matched)

    return {
        "Extracted": extracted,
        "Match": matched,
        "Score": score_val,
        "Explanation": explanation
    }

In [55]:
job_desc = "Looking for Data Scientist with Python, ML, SQL, NLP"

strong_resume = "3 years experience in Python Machine Learning SQL NLP Pandas"
avg_resume = "1 year experience Python SQL"
weak_resume = "Fresher MS Word Communication"

In [56]:
print(run_pipeline(strong_resume, job_desc))
print(run_pipeline(avg_resume, job_desc))
print(run_pipeline(weak_resume, job_desc))

{'Extracted': {'skills': ['Python', 'Machine Learning', 'SQL', 'Pandas', 'NLP'], 'experience': 'present', 'tools': []}, 'Match': {'matched_skills': ['Python', 'Machine Learning', 'SQL', 'Pandas', 'NLP'], 'missing_skills': [], 'match_percentage': 100}, 'Score': 90, 'Explanation': "\nCandidate scored 90 because:\nMatched skills: ['Python', 'Machine Learning', 'SQL', 'Pandas', 'NLP']\nMissing skills: []\nMatch Percentage: 100%\n"}
{'Extracted': {'skills': ['Python', 'SQL'], 'experience': 'present', 'tools': []}, 'Match': {'matched_skills': ['Python', 'SQL'], 'missing_skills': ['Machine Learning', 'Pandas', 'NLP'], 'match_percentage': 40}, 'Score': 40, 'Explanation': "\nCandidate scored 40 because:\nMatched skills: ['Python', 'SQL']\nMissing skills: ['Machine Learning', 'Pandas', 'NLP']\nMatch Percentage: 40%\n"}
{'Extracted': {'skills': [], 'experience': 'none', 'tools': []}, 'Match': {'matched_skills': [], 'missing_skills': ['Python', 'Machine Learning', 'SQL', 'Pandas', 'NLP'], 'match_p